# JARVIS: fine-tuning LoRA (Colab gratuito)
Prima, dall'app o con `POST /api/finetune/export`, fai preparare a JARVIS il dataset: lo salva **cifrato** nel tuo dataset privato Hugging Face.
Qui lo scarichi, lo decifri **solo in memoria** con la tua `ENCRYPTION_KEY` e addestri un adattatore LoRA su un modello aperto piccolo.
Runtime → *Cambia tipo di runtime* → GPU T4. Non incollare mai chiavi in celle di testo: vengono chieste con `getpass`.

In [ ]:
!pip -q install -U transformers peft trl datasets accelerate bitsandbytes huggingface_hub cryptography

In [ ]:
import base64, hashlib, hmac, io, json, getpass
from huggingface_hub import hf_hub_download, HfApi, login
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

HF_TOKEN = getpass.getpass("Token Hugging Face (scrittura): ")
ENC_KEY = getpass.getpass("ENCRYPTION_KEY di JARVIS: ")
login(HF_TOKEN)
me = HfApi().whoami()["name"]
DATASET_REPO = input(f"Dataset privato con il backup (es. {me}/jarvis-private): ").strip()
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # aperto (Apache-2.0), buono in italiano, entra su una T4 in 4 bit
OUT_REPO = f"{me}/jarvis-lora"            # repo MODELLO privato dove salvare l'adattatore

In [ ]:
path = hf_hub_download(DATASET_REPO, "finetune/dataset.jsonl.enc", repo_type="dataset")
blob = open(path, "rb").read()
assert blob[:3] == b"JV1", "formato sconosciuto"
key = hmac.new(ENC_KEY.encode(), b"jarvis-finetune", hashlib.sha256).digest()
plain = AESGCM(key).decrypt(blob[3:15], blob[15:], b"jarvis-backup").decode()   # solo in memoria, mai su disco
rows = [json.loads(l) for l in plain.splitlines() if l.strip()]
print(len(rows), "esempi")

In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
ds = Dataset.from_list(rows).train_test_split(test_size=0.05 if len(rows) >= 40 else 0.2, seed=7)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
cfg = SFTConfig(output_dir="out", num_train_epochs=2, per_device_train_batch_size=2, gradient_accumulation_steps=8,
                learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=5, eval_strategy="epoch",
                save_strategy="no", fp16=True, max_length=2048, report_to="none")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds["train"], eval_dataset=ds["test"], peft_config=lora, processing_class=tok)
trainer.train()

In [ ]:
trainer.model.save_pretrained("jarvis-lora")
tok.save_pretrained("jarvis-lora")
HfApi().create_repo(OUT_REPO, private=True, exist_ok=True)
HfApi().upload_folder(folder_path="jarvis-lora", repo_id=OUT_REPO, commit_message="Adattatore LoRA di JARVIS")
print("Adattatore salvato (privato):", OUT_REPO)

## Dopo l'addestramento
L'adattatore è `OUT_REPO`. Per usarlo serve una GPU che serva il modello base con l'adattatore (ad esempio un endpoint Hugging Face o vLLM su un tuo PC con GPU)
e poi si aggiunge come provider personalizzato (`custom`, URL compatibile OpenAI) in FreeLLMAPI. Render Free non può farlo girare.
Prima di sostituire il router controlla la qualità su domande che conosci.